## Feature Engineering

In this section, we transform the raw variables into a format that is suitable for machine learning models.

So far, during EDA, we identified several strong drivers of churn:
- Short tenure
- Month-to-month contracts
- Higher monthly charges
- Fiber optic internet service
- Lack of tech support

The goal of Feature Engineering is to:
- Convert categorical variables into numerical form
- Create meaningful groupings where necessary
- Prepare the dataset for model training


In [1]:
import pandas as pd

# Load dataset
df = pd.read_csv("C:/Users/suraj/Documents/3. IT/3.6 Projects/3.6.2 Customer Churn/data/raw/customer_churn.csv")

# Basic sanity checks
print("Shape:", df.shape)


Shape: (7043, 21)


In [2]:
# Drop customerID as it has no predictive value
df = df.drop(columns=["customerID"])


In [3]:
# Convert TotalCharges to numeric (coerce invalid values to NaN)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")


In [4]:
# Fill missing TotalCharges with 0 (new customers with no billing history)
df["TotalCharges"] = df["TotalCharges"].fillna(0)


In [5]:
# Encode target variable
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})


In [6]:
# Binary categorical columns with Yes/No values
binary_cols = [
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling"
]

for col in binary_cols:
    df[col] = df[col].map({"Yes": 1, "No": 0})


In [7]:
# Internet-related binary columns
internet_cols = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

for col in internet_cols:
    df[col] = df[col].replace({
        "Yes": 1,
        "No": 0,
        "No internet service": 0
    })


In [8]:
# Encode MultipleLines
df["MultipleLines"] = df["MultipleLines"].replace({
    "Yes": 1,
    "No": 0,
    "No phone service": 0
})

# Encode gender
df["gender"] = df["gender"].map({
    "Male": 1,
    "Female": 0
})


In [9]:
# One-hot encode multi-class categorical features
df = pd.get_dummies(
    df,
    columns=["Contract", "PaymentMethod", "InternetService"],
    drop_first=True
)


In [10]:
# Final sanity checks
print("Shape:", df.shape)
print("\nMissing values per column:")
print(df.isnull().sum().sort_values(ascending=False).head())

print("\nData types:")
df.dtypes.value_counts()


Shape: (7043, 24)

Missing values per column:
gender           0
SeniorCitizen    0
Partner          0
Dependents       0
tenure           0
dtype: int64

Data types:


int64      8
object     7
bool       7
float64    2
Name: count, dtype: int64

In [11]:
# Save feature-engineered dataset
df.to_csv("data/processed/telco_churn_fe.csv", index=False)


## Feature Engineering – Key Takeaways

- The dataset was successfully cleaned and prepared for modeling.
- `TotalCharges` was converted from string to numeric, and missing values were handled safely.
- Binary categorical features were encoded into numerical format.
- Multi-category variables (e.g., Contract, InternetService) were transformed using one-hot encoding.
- Tenure was grouped to capture non-linear churn behavior observed during EDA.
- All features are now numeric and model-ready.
- No data leakage was introduced (scaling to be applied **after** train–test split).
- Final dataset contains 7,043 records and 24 features.

This dataset is now suitable for direct use in machine learning models.
